# Ticket Support Data Masking & Privacy-Preserving Data Preparation

This Jupyter Notebook documents the data preparation, transformation, and sensitive-data masking process applied to ticket support data handled during my internship. The source data was collected from Redmine and Freshdesk and was used to support ticket monitoring, operational reporting, and the development of a Power BI ticket analytics dashboard.

The notebook demonstrates how raw support-ticket data can be prepared for analytics while protecting confidential business and personal information. The process includes data cleaning, standardization, identifier masking, client anonymization, consultant/agent anonymization, ticket-ID masking, contact-information masking, and the creation of derived analytical fields while preserving the relationships and structure required for reporting.

The resulting dataset is intended for portfolio and demonstration purposes only. Sensitive client names, employee/consultant information, contact details, ticket identifiers, and other confidential information have been replaced or masked to prevent disclosure of the original organizational data.

This project demonstrates practical skills developed during my internship in data preparation, data privacy and anonymization, Python/Pandas, exploratory data analysis (EDA), data transformation, and business intelligence reporting, particularly in preparing operational support data for Power BI visualization and analysis.

# Importing Libraries

In [ ]:
# pandas is used for loading, cleaning, transforming, and analyzing tabular ticket/client data through DataFrames.
import pandas as pd
# NumPy provides numerical and array-based operations that support calculations and data manipulation.
import numpy as np
# The re (regular expression) module is used to search, replace, and clean text patterns such as names and ticket subjects.
import re
# Import the required Python module for the operations below.
import hashlib
# Import defaultdict, which automatically creates a default value for missing dictionary keys.
from collections import defaultdict

# Importing Excel Dataset

In [ ]:
# Store the input Excel filename in a variable so the source file can be changed without rewriting the loading code.
RM_FD_Dataset = pd.ExcelFile(r"RM_FD_Dataset.xlsx")

# Read the workbook's worksheets into pandas DataFrames so they can be cleaned and transformed programmatically.
df = pd.read_excel(RM_FD_Dataset, sheet_name=None)

## Client Environment

### Exploratory Data Analysis

In [ ]:
# .info() summarizes rows, columns, non-null values, and data types to understand the structure of the client-environment data.
df["Client Envi V2"].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 145 entries, 0 to 144
Data columns (total 6 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   CLIENT NAME FOR SEARCH  145 non-null    object
 1   ENVIRONMENT             145 non-null    object
 2   CLIENT STATE            145 non-null    object
 3   PHASES                  121 non-null    object
 4   PRODUCT                 145 non-null    object
 5   CLIENT FULL NAME        145 non-null    object
dtypes: object(6)
memory usage: 6.9+ KB


### Output analysis
- The **Client Envi V2** table contains **145 rows and 6 columns**.
- All six columns are stored as `object`, so they are categorical/text-like fields rather than numeric fields.
- `PHASES` has **121 non-null values**, meaning **24 rows are missing a phase**.
- The other five columns are complete in this output, with 145 non-null values each.
- This confirms that the client-environment table is small and suitable for dictionary-based anonymization.

In [5]:
# Copying raw data of df["Client Envi V2"] into another df to be masked
client_environment_df = df["Client Envi V2"].copy()

### Masking Client Names

In [ ]:
# Identify distinct values needed for mapping, validation, or anonymization.

clients = df["Client Envi V2"]["CLIENT NAME FOR SEARCH"].unique()

In [ ]:
# Assigning each unique client name a masking name such as Client 001, so on, so forth
# Create a dictionary that maps each original client identifier to a portfolio-safe alias such as Client 001.
client_alias = {
    client: f"Client {i:03d}"
    for i, client in enumerate(clients, start=1)
}

In [ ]:
# Mapping the client names for the masking to occur

client_environment_df["CLIENT NAME FOR SEARCH"] = (
    df["Client Envi V2"]["CLIENT NAME FOR SEARCH"]
    .map(client_alias)
)

# Since the "CLIENT NAME FOR SEARCH" is the unique classifier of the client's name, 
# we're setting another full name on "CLIENT FULL NAME".

# Assign the transformed result to the specified DataFrame column.
client_environment_df["CLIENT FULL NAME"] = (
    "Full Name of " + client_environment_df["CLIENT NAME FOR SEARCH"]
)

client_environment_df

,CLIENT NAME FOR SEARCH,ENVIRONMENT,CLIENT STATE,PHASES,PRODUCT,CLIENT FULL NAME
0,Client 001,On Cloud,Active,Go-Live,PayrollPlus,Full Name of Client 001
1,Client 002,On Premise,Active,Go-Live,PayrollPro,Full Name of Client 002
2,Client 003,On Premise,Active,Support Handover,PayrollPlus,Full Name of Client 003
3,Client 004,On Premise,Active,Go-Live,PayrollPlus,Full Name of Client 004
4,Client 005,On Cloud,Active,Parallel Run,PayrollPlus,Full Name of Client 005
...,...,...,...,...,...,...
140,Client 140,On Cloud,Active,Support Handover,NetSuite,Full Name of Client 140
141,Client 141,On Cloud,Active,Support Handover,NetSuite,Full Name of Client 141
142,Client 142,On Cloud,Active,Support Handover,Monday.com,Full Name of Client 142
143,Client 143,On Premise,Active,Support Handover,PayrollPlus,Full Name of Client 143


### Output analysis
- The client identifiers have been replaced with sequential aliases such as **Client 001, Client 002, ...**.
- `CLIENT FULL NAME` is derived consistently as **Full Name of Client XXX**.
- Non-sensitive business attributes such as environment, state, phase, and product remain available for analytics.
- The output demonstrates that the masking preserves the table's structure while replacing identifying client names.

In [9]:
# Checking for duplicates, should be done at the start but contains real client name

client_environment_df["CLIENT NAME FOR SEARCH"].value_counts().loc[lambda s: s > 1]

CLIENT NAME FOR SEARCH
Client 089    2
Name: count, dtype: int64

### Output analysis
- The duplicate check found **Client 089 occurring twice**.
- This means the masking itself did not guarantee one row per client record; duplicate rows existed in the source/result.
- The next cell removes exact duplicate rows and repeats the check.

In [10]:
client_environment_df = client_environment_df.drop_duplicates()
client_environment_df["CLIENT NAME FOR SEARCH"].value_counts().loc[lambda s: s > 1]

Series([], Name: count, dtype: int64)

# Ticket Data

## Exploratory Data Analysis

### Info

In [ ]:
# Display the DataFrame structure, non-null counts, data types, and memory usage.
df["Redmine Raw"].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3204 entries, 0 to 3203
Data columns (total 22 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   Unnamed: 0            3204 non-null   int64         
 1   Start date            3204 non-null   datetime64[ns]
 2   Due date              3181 non-null   datetime64[ns]
 3   Project               3204 non-null   object        
 4   Subject               3204 non-null   object        
 5   Status                3204 non-null   object        
 6   Priority              3204 non-null   object        
 7   Author                3204 non-null   object        
 8   Assignee              3204 non-null   object        
 9   Updated               3204 non-null   datetime64[ns]
 10  Last updated by       3192 non-null   object        
 11  Closed                3028 non-null   datetime64[ns]
 12  Tracker               3204 non-null   object        
 13  Bug ID            

### Output analysis
- The Redmine raw dataset contains **3,204 ticket records** and **22 columns**.
- Most fields are complete, but some columns contain missing values, including `Due date`, `Last updated by`, `Closed`, and `Bug ID`.

In [ ]:
# Display the DataFrame structure, non-null counts, data types, and memory usage.
df["FD Raw"].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1807 entries, 0 to 1806
Data columns (total 29 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   Ticket ID                     1807 non-null   int64         
 1   Subject                       1807 non-null   object        
 2   Status                        1807 non-null   object        
 3   Priority                      1807 non-null   object        
 4   Type                          1534 non-null   object        
 5   Agent                         1807 non-null   object        
 6   Group                         1807 non-null   object        
 7   Created time                  1807 non-null   datetime64[ns]
 8   Due by Time                   1807 non-null   datetime64[ns]
 9   Resolved time                 1774 non-null   datetime64[ns]
 10  Closed time                   799 non-null    datetime64[ns]
 11  Last update time              

### Output analysis
- The Freshdesk raw dataset contains **1,807 ticket records** and **29 columns**.
- Several operational fields are available, including ticket ID, subject, status, priority, agent, group, dates, response/resolution information, contact information, and company information.

### Columns

In [ ]:
# Display the current column names for schema verification.
df["Redmine 2022-2026"].columns

Index(['DATE CREATED', 'START DATE', 'DUE DATE', 'AGING', 'CLIENT',
       'TITLE CONCERN / SUBJECT', 'AUTHOR', 'ASSIGNEE', 'TICKET ID',
       'REDMINE LINK', 'STATUS', 'PRIORITY', 'DAYS DIFF', 'DUE STATUS',
       'LAST UPDATED BY', 'DATE UPDATED', 'DATE CLOSED', 'C.DURATION',
       'C.DURATION (IN WORDS)', 'R.DURATION', 'R.DURATION (IN WORDS)',
       'ENVIRONMENT', 'CLIENT STATUS', 'MODULE', 'SPENT TIME', 'ISSUE',
       'SUPPORT LEVEL', 'SEVERITY'],
      dtype='object')

### Output analysis
- The displayed Redmine processed schema contains **28 columns**.
- These columns include identifiers, dates, aging, client information, ticket status, due-status measures, durations, environment/client attributes, module information, and support-level information.
- This confirms that the processed Redmine table is designed as an analytics-ready structure rather than a copy of the raw source.

In [ ]:
# Display the current column names for schema verification.
df["FD Raw"].columns

Index(['Ticket ID', 'Subject', 'Status', 'Priority', 'Type', 'Agent', 'Group',
       'Created time', 'Due by Time', 'Resolved time', 'Closed time',
       'Last update time', 'Initial response time', 'Time tracked',
       'First response time (in hrs)', 'Resolution time (in hrs)',
       'Agent interactions', 'Customer interactions', 'Resolution status',
       'First response status', 'Tags', 'Implementation Phase',
       'Support Level', 'Assigned To', 'Full name', 'Email', 'Tags.1',
       'Contact ID', 'Company Name'],
      dtype='object')

### Output analysis
- The Freshdesk source contains **29 columns**, including contact, ticket, support, company, and timing information.
- These columns provide the source fields needed for the later processed Freshdesk dataset.

### Copy

In [ ]:
# Create an independent copy so subsequent masking does not modify the original DataFrame.
redmine_raw_df = df["Redmine Raw"].copy()
freshdesk_raw_df = df["FD Raw"].copy()

### Masking Ticket IDs

In [ ]:
# Define the function create_ticket_alias used to mask the ticket number 
def create_ticket_alias(ticket_ids, platform):
    ticket_ids = sorted(pd.Series(ticket_ids).dropna().unique())

    return {
        ticket_id: f"{platform}-TKT-{i:04d}"
        for i, ticket_id in enumerate(ticket_ids, start=1)
    }

In [ ]:
# Assign the computed result to rm_ticket_alias.
redmine_raw_df["Unnamed: 0"] = (
    redmine_raw_df["Unnamed: 0"]
    .map(create_ticket_alias(
                df["Redmine Raw"]["Unnamed: 0"],
                "RM"
            )
        )
    )

In [ ]:
# Assign the computed result to fd_ticket_alias.
freshdesk_raw_df["Ticket ID"] = (
    freshdesk_raw_df["Ticket ID"]
    .map(
        create_ticket_alias(
            df["FD Raw"]["Ticket ID"],
            "FD"
        )
    )
)

### Masking Client Names

Since we have the client's unique classifier on ["CLIENT NAME FOR SEARCH"] and the client's full name on ["CLIENT FULL NAME"] in the df["Client Envi V2"], implying that df["Client Envi V2"] is a manually maintained lookup table, we'll just simplying it by using substring matching.

However, other methods such as fuzzy matching is also welcome to be used.

In [ ]:
# So we'll first create project alias as a lookup table for project masking. 
project_alias = {}

# Assign the computed result to projects.
projects = pd.concat([
    df["Redmine Raw"]["Project"].dropna(),
    df["FD Raw"]["Company Name"].dropna().astype(str)
], ignore_index=True)

# Find the highest existing alias number
numbers = [
    int(re.search(r"(\d+)$", alias).group(1))
    for alias in client_alias.values()
]

next_id = max(numbers) + 1 if numbers else 1

''' 
After that, we're traversing each unique project/client name present in Redmine Raw Data.
Then, we'll check if the substrings present in [Client Envi V2] matches to the 
project names in Redmine. If it is, we'll replace it with a masked value.

'''

# The purpose is to match ticket-system clients with the existing Client Environment reference list.
for project in sorted(projects.unique()):

    # Track whether the current project was successfully matched to an existing client.
    matched = False

    # Compare the current project against every known client in the existing client alias mapping.
    for client in client_alias:

        # Check whether the known client name appears within the project name.
        # This allows projects containing additional platform/environment information
        # to still be associated with the correct client.
        if str(client).lower() in project.lower():
            project_alias[project] = client_alias[client]
            matched = True
            break

    # If the project does not match any client currently present in Client Environment,
    # treat it as a newly discovered client from the Redmine/Freshdesk ticket data.
    if not matched:
        alias = f"Client {next_id:03d}"

        # Generate a new anonymized client identifier for the newly discovered client.
        project_alias[project] = alias

        # Add the newly discovered project/client to the project-to-client mapping.
        client_alias[project] = alias
        
        # Add the newly discovered client to the Client Environment dataset.
        # This keeps the client reference table synchronized with clients
        # encountered in Redmine/Freshdesk.
        #
        # Environment, client state, phase, and product are left blank because
        # these attributes are not available from the ticket project information.
        client_environment_df.loc[next_id] = {
            "CLIENT NAME FOR SEARCH": alias,
            "CLIENT FULL NAME": "Full Name of " + alias,
            "ENVIRONMENT": None,
            "CLIENT STATE": None,
            "PHASES": None,
            "PRODUCT": None
        }

        next_id += 1

### Output analysis
- The client mapping table has expanded to **216 rows**.
- The additional rows (`Client 212` through `Client 216` in the displayed tail) were created for project/company values that were not already matched to an existing client alias.
- Their environment, client state, phase, and product are `None`, which is intentional for newly discovered identifiers that lack corresponding client-environment metadata.

In [25]:
print("Redmine Client Values: ", redmine_raw_df["Project"].isna().sum())
print("Freshdesk Client Values: ", freshdesk_raw_df["Company Name"].isna().sum())

Redmine Client Values:  0
Freshdesk Client Values:  619


### Output analysis
- Redmine has **0 missing project/client values**.
- Freshdesk has **619 missing company values** before the temporary `Unknown Client` replacement.
- This is an important data-quality difference between the two source systems and explains why Freshdesk later contains `Unknown Client` records.

In [ ]:
# Replace the specified values or text patterns with their masked or standardized equivalents.
redmine_raw_df["Project"] = redmine_raw_df["Project"].replace(project_alias)

In [ ]:
# Temporary client name for null values
# Assign the transformed result to the specified DataFrame column.
freshdesk_raw_df["Company Name"] = (
    freshdesk_raw_df["Company Name"]
    .fillna("Unknown Client")
)

# Replace the specified values or text patterns with their masked or standardized equivalents.
freshdesk_raw_df["Company Name"] = freshdesk_raw_df["Company Name"].replace(project_alias)

### Masking Ticket Subjects

In [ ]:
# In order to mask the ticket subjects, there are many ways to do it such as assigning an ID number to each, but the one I'll use is hashing using hashlib.

# Ticket subjects may contain sensitive or identifiable information.
# Instead of assigning sequential IDs, SHA-256 hashing is used to generate
# a consistent anonymized representation of the subject text.

import hashlib
import re

'''
In the other subjects, there are labels in order to determine which support levels are they. 
Hence, we'll extract it if it exists, then we'll hash the remaining values.
'''

def anonymize_subject(subject):
    if pd.isna(subject):
        return subject

    # Assign the computed result to subject.
    subject = str(subject)

    # Search for a support-level label at the beginning of the subject.
    # The pattern identifies labels such as [L1], [L2], [L3], etc.,
    # and captures the remaining subject description separately.
    match = re.match(r"\[(L\d+).*?\]\s*(.*)", subject)

    # If a support-level label is found, process the label and description separately.
    if match:
        level = match.group(1)
        description = match.group(2)

        # Convert the text to bytes so it can be processed by the hashing function.
        hashed = hashlib.sha256(description.encode()).hexdigest()[:8]

        return f"[{level}] {hashed}"

    # If the subject does not contain an [L*] support-level label,
    # hash the entire subject because there is no classification label
    # that needs to be preserved.

    return hashlib.sha256(subject.encode()).hexdigest()[:8]

In [ ]:
# Apply the specified function or transformation to each value or row.
redmine_raw_df["Subject"] = redmine_raw_df["Subject"].apply(anonymize_subject)
freshdesk_raw_df["Subject"] = freshdesk_raw_df["Subject"].apply(anonymize_subject)

### Masking Consultants

In [ ]:
# The function standardizes the formatting of names before they are
# used in the subsequent masking and data-processing steps.
def normalize_name(name):
    if pd.isna(name):
        return name

    name = str(name)

    # Freshdesk represents spaces in some consultant names using underscores.
    # Replace underscores with regular spaces to recover a standardized
    # readable name format.
    #
    # Example:
    # "John_Doe" → "John Doe"
    name = name.replace("_", " ")

    # Replace consecutive whitespace characters with a single space
    # and remove unnecessary spaces from the beginning and end.
    name = re.sub(r"\s+", " ", name).strip()

    return name



In [ ]:
// Apply the normalize_name function to every value containing consultants' names
consultants = pd.unique(
    pd.concat([
        df["Redmine Raw"]["Author"],
        df["Redmine Raw"]["Assignee"],
        df["Redmine Raw"]["Last updated by"],
        df["FD Raw"]["Assigned To"]
    ], ignore_index=True)
    .dropna()
    .map(normalize_name)
)

In [ ]:
# Create a dictionary that maps each unique consultant name
# to an anonymized consultant identifier.
consultant_alias = {
    name: f"Consultant {i:03d}"
    for i, name in enumerate(consultants, start=1) 
}

In [ ]:
# Replace the original author names with their corresponding
# anonymized consultant aliases.

redmine_raw_df["Author"] = redmine_raw_df["Author"].apply(
    lambda x: consultant_alias.get(x, x)
)
redmine_raw_df["Assignee"] = redmine_raw_df["Assignee"].apply(
    lambda x: consultant_alias.get(x, x)
)
redmine_raw_df["Last updated by"] = redmine_raw_df["Last updated by"].apply(
    lambda x: consultant_alias.get(x, x)
)
freshdesk_raw_df["Assigned To"] = freshdesk_raw_df["Assigned To"].apply(
    lambda x: consultant_alias.get(x, x)
)

### Masking Product Types

In [ ]:
# Create a mapping between each unique Freshdesk ticket type
# and an anonymized product identifier.
product_type_alias = {
    value: f"Product {i:03d}"
    for i, value in enumerate(
        freshdesk_raw_df["Type"].dropna().unique(),
        start=1
    )
}

# Replace the original Freshdesk ticket types with their
# corresponding anonymized product identifiers.
freshdesk_raw_df["Type"] = freshdesk_raw_df["Type"].map(product_type_alias)

### Masking Company Agents

In [ ]:
# Create a mapping between each unique Freshdesk agent
# and an anonymized support team identifier.
agent_alias = {
    value: f"Support Team {i:03d}"
    for i, value in enumerate(
        freshdesk_raw_df["Agent"].dropna().unique(),
        start=1
    )
}

# Replace the original Freshdesk agent values with their
# corresponding anonymized support team identifiers.
freshdesk_raw_df["Agent"] = freshdesk_raw_df["Agent"].map(agent_alias)

### Masking Support Groups

In [ ]:
# Create a mapping between each unique Freshdesk group
# and an anonymized group identifier.
group_alias = {
    value: f"Group {i:03d}"
    for i, value in enumerate(
        freshdesk_raw_df["Group"].dropna().unique(),
        start=1
    )
}

# Replace the original Freshdesk group values with
# their corresponding anonymized group identifiers.
freshdesk_raw_df["Group"] = freshdesk_raw_df["Group"].map(group_alias)

### Masking Ticket Tags

In [ ]:
# Define a function to mask sensitive or identifying values
# within the Freshdesk Tags field.
def mask_tags(value):
    if pd.isna(value):
        return value

    for old, new in tag_mapping.items():
        # Replace the original tag with its standardized or anonymized value.
        value = value.replace(old, new)

    return value

# Apply the tag-masking function to every value in the Freshdesk Tags column.
freshdesk_raw_df["Tags"] = freshdesk_raw_df["Tags"].apply(mask_tags)

### Masking Contact Emails

In [ ]:
# Check whether every value in the "Email" column is exactly the same
# as the corresponding value in the "Contact ID" column.
(freshdesk_raw_df["Email"] == freshdesk_raw_df["Contact ID"]).all()

np.True_

In [ ]:
# Create an empty dictionary to store the mapping between original emails and generated aliases.
email_alias = {}

# Create a counter dictionary where each company/domain starts counting from 0.
company_counter = defaultdict(int)

# Loop through every row in the Freshdesk DataFrame.
for _, row in freshdesk_raw_df.iterrows():

    email = row["Email"]

    if pd.isna(email):
        continue

    # Get the company name from the current row.
    company = row["Company Name"]

    if pd.isna(company):
        # Use "example" as the default domain when the company name is missing.
        domain = "example"
    else:
        # Convert the company name to a string, remove all characters
        # except letters and numbers, and convert the result to lowercase.
        domain = re.sub(r"[^a-zA-Z0-9]", "", str(company)).lower()

        if not domain:
            domain = "example"

    # Check whether this email has already been assigned an alias.
    if email not in email_alias:
        company_counter[domain] += 1

        # Create a unique anonymized email alias.
        # :03d formats the counter as a three-digit number, e.g. 001, 002, 003.
        email_alias[email] = (
            f"user{company_counter[domain]:03d}@{domain}.com"
        )

# Replace every original email in the DataFrame with its generated anonymized alias.
freshdesk_raw_df["Email"] = freshdesk_raw_df["Email"].map(email_alias)
freshdesk_raw_df["Contact ID"] = freshdesk_raw_df["Email"]

### Masking Contact Names

In [ ]:
# Create an empty dictionary that will store the mapping
# between each company and its email/contact alias.
client_contact_alias = {}

# Group the Freshdesk data by "Company Name".
# Each "client" is a company name, and "group" contains all rows
# belonging to that company.
for client, group in freshdesk_raw_df.groupby("Company Name"):

    # Get all unique, non-empty email addresses for the current company.
    contacts = (
        group["Email"]
        .dropna()
        .unique()
    )

    for i, email in enumerate(contacts, start=1):
        # Create an alias for the combination of company and email.
        client_contact_alias[(client, email)] = (
            f"{client} Rep {i:03d}"
        )

In [46]:
client_contact_alias

{('Client 001', 'user001@client001.com'): 'Client 001 Rep 001',
 ('Client 003', 'user001@client003.com'): 'Client 003 Rep 001',
 ('Client 004', 'user001@client004.com'): 'Client 004 Rep 001',
 ('Client 005', 'user001@client005.com'): 'Client 005 Rep 001',
 ('Client 005', 'user002@client005.com'): 'Client 005 Rep 002',
 ('Client 006', 'user001@client006.com'): 'Client 006 Rep 001',
 ('Client 007', 'user001@client007.com'): 'Client 007 Rep 001',
 ('Client 007', 'user002@client007.com'): 'Client 007 Rep 002',
 ('Client 007', 'user003@client007.com'): 'Client 007 Rep 003',
 ('Client 008', 'user001@client008.com'): 'Client 008 Rep 001',
 ('Client 010', 'user001@client010.com'): 'Client 010 Rep 001',
 ('Client 011', 'user001@client011.com'): 'Client 011 Rep 001',
 ('Client 011', 'user002@client011.com'): 'Client 011 Rep 002',
 ('Client 011', 'user003@client011.com'): 'Client 011 Rep 003',
 ('Client 014', 'user001@client014.com'): 'Client 014 Rep 001',
 ('Client 014', 'user002@client014.com')

In [ ]:
# Create or update the "Full name" column in the Freshdesk DataFrame.
freshdesk_raw_df["Full name"] = freshdesk_raw_df.apply(
    lambda row: client_contact_alias.get(
        # Use Company Name + Email as the lookup key.
        (row["Company Name"], row["Email"]),
        # If the key is not found, keep the original Full name.
        row["Full name"]
    ),
    axis=1
)

### Masking Bug IDs

In [ ]:
def mask_bug_id(value, platform):
    if pd.isna(value):
        return pd.NA

    # Convert the Bug ID to a string, encode it as bytes,
    # create a SHA-256 hash, take the first 8 characters,
    # and convert them to uppercase.
    digest = hashlib.sha256(str(value).encode()).hexdigest()[:8].upper()

    # Create the masked Bug ID using the platform prefix
    return f"{platform}-BG-{digest}"

# Apply the mask_bug_id function to every value in the "Bug ID" column.
redmine_raw_df["Bug ID"] = redmine_raw_df["Bug ID"].apply(
    lambda x: mask_bug_id(x, "RM")
)

### Masking Root Causes

In [ ]:
# Clean and standardize the "Root Cause" column.
redmine_raw_df["Root Cause"] = (
    redmine_raw_df["Root Cause"]

    # Remove text enclosed in parentheses, including any spaces before it.
    .str.replace(r"\s*\(.*?\)", "", regex=True)

    # Replace multiple consecutive whitespace characters with a single space.
    .str.replace(r"\s+", " ", regex=True)

    # Remove leading and trailing spaces from each value.
    .str.strip()
)

## Creating Processed dataframe

In [ ]:
# Define the reference date as May 30, 2026.
reference_date = pd.Timestamp("2026-05-30")
reference_date

Timestamp('2026-05-30 00:00:00')

In [ ]:
# Set "CLIENT NAME FOR SEARCH" as the index of the DataFrame.
client_environment_df = client_environment_df.set_index("CLIENT NAME FOR SEARCH")
client_environment_df.columns

Index(['ENVIRONMENT', 'CLIENT STATE', 'PHASES', 'PRODUCT', 'CLIENT FULL NAME'], dtype='object')

In [ ]:
def format_duration(td):
    # Check whether the duration value is missing.
    if pd.isna(td):
        # Return None when the duration is missing.
        return None

    # Extract the number of complete days from the timedelta.
    days = td.days

    # Extract the number of hours remaining after the days.
    hours = td.components.hours

    # Extract the number of minutes remaining after the hours.
    minutes = td.components.minutes

    # Return the duration as a readable text string.
    return f"{days} days, {hours} hours {minutes} minutes"

In [54]:
redmine_processed_df = pd.DataFrame({
    "DATE CREATED": redmine_raw_df["Created"],
    "START DATE": redmine_raw_df["Start date"].dt.date,
    "DUE DATE": redmine_raw_df["Due date"].dt.date,

    "AGING": (
        redmine_raw_df["Closed"]
        .where(
            redmine_raw_df["Status"].str.lower().eq("closed"),
            reference_date
        )
        .sub(redmine_raw_df["Created"])
        .dt.days
    ),

    "CLIENT": redmine_raw_df["Project"].map(client_environment_df["CLIENT FULL NAME"]),
    "TITLE CONCERN / SUBJECT": redmine_raw_df["Subject"],
    "AUTHOR": redmine_raw_df["Author"],
    "ASSIGNEE": redmine_raw_df["Assignee"],
    "TICKET ID": redmine_raw_df["Unnamed: 0"],

    "REDMINE LINK": (
        "https://" + redmine_raw_df["Project"].map(client_environment_df["CLIENT FULL NAME"]).str.lower().str.replace(" ", "", regex=False) + ".redmine.com/a/tickets/"
        + redmine_raw_df["Unnamed: 0"].astype(str)
    ),
    "STATUS": redmine_raw_df["Status"],
    "PRIORITY": redmine_raw_df["Priority"],

    "DAYS DIFF": (
        (reference_date - redmine_raw_df["Due date"])
        .dt.days
        .where(
            ~redmine_raw_df["Status"].str.lower().eq("closed"),
            None
        ).astype("Int64")
    ),

    "DUE STATUS": np.select(
    [
        redmine_raw_df["Status"].str.lower().eq("closed"),
        (redmine_raw_df["Status"].str.lower().ne("closed")) &
        (redmine_raw_df["Due date"] < reference_date)
    ],
    [
        "Done",
        "Overdue"
    ],
    default=None),

    "LAST UPDATED BY": redmine_raw_df["Last updated by"],
    "DATE UPDATED": redmine_raw_df["Updated"],
    "DATE CLOSED": redmine_raw_df["Closed"],

    "C.DURATION": (redmine_raw_df["Closed"]
        .where(
            redmine_raw_df["Status"].str.lower().eq("closed")
        ).sub(redmine_raw_df["Created"])
        .dt.total_seconds()
        / 86400
        ).round(4),
    "C.DURATION (IN WORDS)": redmine_raw_df["Closed"]
        .sub(redmine_raw_df["Created"])
        .apply(format_duration)
    ,

    "R.DURATION": (redmine_raw_df["Updated"]
        .where(
            redmine_raw_df["Status"].str.lower().eq("resolved")
        ).sub(redmine_raw_df["Created"])
        .dt.total_seconds()
        / 86400
    ).round(4),
    "R.DURATION (IN WORDS)": (redmine_raw_df["Updated"]
        .where(
            redmine_raw_df["Status"].str.lower().eq("resolved")
        ).sub(redmine_raw_df["Created"])
        .apply(format_duration)
        ),

    "ENVIRONMENT": redmine_raw_df["Project"].map(client_environment_df["ENVIRONMENT"]),
    "CLIENT STATUS": redmine_raw_df["Project"].map(client_environment_df["CLIENT STATE"]),
    "PROJECT PHASE": redmine_raw_df["Project"].map(client_environment_df["PHASES"]),
    "CLIENT PRODUCT": redmine_raw_df["Project"].map(client_environment_df["PRODUCT"]).map(product_type_alias),

    "MODULE": redmine_raw_df["Module"],
    "SPENT TIME": redmine_raw_df["Spent time"],

    "SUPPORT LEVEL": redmine_raw_df["Subject"].str.extract(r"\[([^\]]+)\]", expand=False),
})

In [55]:
redmine_processed_df

,DATE CREATED,START DATE,DUE DATE,AGING,CLIENT,TITLE CONCERN / SUBJECT,AUTHOR,ASSIGNEE,TICKET ID,REDMINE LINK,...,C.DURATION (IN WORDS),R.DURATION,R.DURATION (IN WORDS),ENVIRONMENT,CLIENT STATUS,PROJECT PHASE,CLIENT PRODUCT,MODULE,SPENT TIME,SUPPORT LEVEL
0,2026-03-19 13:12:00,2026-03-19,2026-03-24,71,Full Name of Client 087,[L2] 900f9b9d,Consultant 001,Consultant 005,RM-TKT-3204,https://fullnameofclient087.redmine.com/a/tick...,...,None,NaN,None,On Cloud,Active,Support Handover,Product 001,Job Scheduler,0,L2
1,2026-03-19 10:48:00,2026-03-19,2026-03-20,5,Full Name of Client 031,[L1] 53f353f6,Consultant 002,Consultant 002,RM-TKT-3203,https://fullnameofclient031.redmine.com/a/tick...,...,"5 days, 20 hours 47 minutes",NaN,None,On Cloud,Active,Support Handover,Product 001,"Offboarding, Onboarding",0,L1
2,2026-03-19 06:27:00,2026-03-19,2026-03-19,18,Full Name of Client 082,[L2] 2fe0c9e9,Consultant 001,Consultant 005,RM-TKT-3202,https://fullnameofclient082.redmine.com/a/tick...,...,"18 days, 1 hours 15 minutes",NaN,None,On Cloud,Active,Support Handover,Product 001,Employee Information Manager (EIM),0,L2
3,2026-03-19 06:22:00,2026-03-19,2026-03-19,0,Full Name of Client 107,[L2] 47986223,Consultant 003,Consultant 005,RM-TKT-3201,https://fullnameofclient107.redmine.com/a/tick...,...,"0 days, 0 hours 53 minutes",NaN,None,On Cloud,Active,UAT,Product 001,Attendance,0,L2
4,2026-03-18 12:25:00,2026-03-18,2026-03-18,8,Full Name of Client 107,[L3] 85dcf6f6,Consultant 003,Consultant 017,RM-TKT-3200,https://fullnameofclient107.redmine.com/a/tick...,...,"8 days, 1 hours 52 minutes",NaN,None,On Cloud,Active,UAT,Product 001,Employee Information Manager (EIM),0,L3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3199,2023-01-30 09:43:00,2023-01-30,2023-02-01,7,Full Name of Client 022,6b7d2e56,Consultant 005,Consultant 032,RM-TKT-0005,https://fullnameofclient022.redmine.com/a/tick...,...,"7 days, 1 hours 26 minutes",NaN,None,On Cloud,Inactive,NaN,Product 001,Recruitment,0,NaN
3200,2023-01-26 05:34:00,2023-01-26,2023-01-27,6,Full Name of Client 022,677ac977,Consultant 005,Consultant 032,RM-TKT-0004,https://fullnameofclient022.redmine.com/a/tick...,...,"6 days, 12 hours 56 minutes",NaN,None,On Cloud,Inactive,NaN,Product 001,Recruitment,0,NaN
3201,2023-01-16 16:27:00,2023-01-16,2023-01-18,1,Full Name of Client 088,9095d1ce,Consultant 005,Consultant 032,RM-TKT-0003,https://fullnameofclient088.redmine.com/a/tick...,...,"1 days, 0 hours 58 minutes",NaN,None,On Premise,Active,Parallel Run,Product 001,Employee Information,0,NaN
3202,2022-11-23 06:19:00,2022-11-23,2022-11-30,170,Full Name of Client 026,0e286b4f,Consultant 001,Consultant 022,RM-TKT-0002,https://fullnameofclient026.redmine.com/a/tick...,...,"170 days, 3 hours 52 minutes",NaN,None,On Premise,Active,Go-Live,Product 001,Offboarding,0,NaN


In [56]:
freshdesk_processed_df = pd.DataFrame({

    "CREATED DATE": freshdesk_raw_df["Created time"].dt.date,

    "DUE DATE": freshdesk_raw_df["Due by Time"].dt.date,

    "AGING": (
        freshdesk_raw_df["Closed time"]
        .where(
            freshdesk_raw_df["Status"].str.lower().eq("closed"),
            reference_date
        )
        .sub(freshdesk_raw_df["Created time"])
        .dt.days
    ),

    "DAY DIFFERENCE": (
        abs(freshdesk_raw_df["Due by Time"]
        .sub(reference_date)
        .dt.days)
    ),

    "TITLE CONCERN": freshdesk_raw_df["Subject"],

    "CLIENT EMAIL": freshdesk_raw_df["Email"],

    "COMPANY NAME": freshdesk_raw_df["Company Name"].map(client_environment_df["CLIENT FULL NAME"]).fillna("Unknown Client"),

    "TICKET ID": freshdesk_raw_df["Ticket ID"],

    "FRESH DESK LINK": (
        "https://" + freshdesk_raw_df["Company Name"].str.lower().str.replace(" ", "", regex=False) + ".freshdesk.com/a/tickets/"
        + freshdesk_raw_df["Ticket ID"].astype(str)
    ),

    "IN-CHARGE": freshdesk_raw_df["Assigned To"].fillna("Unnamed Consultant"),

    "NO OF HOURS SPENT": freshdesk_raw_df["Time tracked"],

    "INITIAL RESPONSE DATE": freshdesk_raw_df["Initial response time"],

    "RESOLVED DATE": freshdesk_raw_df["Resolved time"],

    "CLOSED DATE": freshdesk_raw_df["Closed time"],

    "LAST UPDATED DATE": freshdesk_raw_df["Last update time"],

    "TYPE": freshdesk_raw_df["Type"],

    "GROUP": freshdesk_raw_df["Group"],

    "PRIORITY": freshdesk_raw_df["Priority"],

    "TICKET STATUS": freshdesk_raw_df["Status"],

    "DUE STATUS": np.select(
    [
        freshdesk_raw_df["Status"].str.lower().eq("closed"),
        (freshdesk_raw_df["Status"].str.lower().ne("closed")) &
        (freshdesk_raw_df["Due by Time"] < reference_date)
    ],
    [
        "Done",
        "Overdue"
    ],
    default=None),

    "INITIAL RESPONSE STATUS": freshdesk_raw_df["First response status"],

    "RESOLUTION STATUS": freshdesk_raw_df["Resolution status"],

    "RESOLUTION TIME DIFFERENCE": np.where(
        freshdesk_raw_df["Resolved time"].isna(),
        0,
        (
            freshdesk_raw_df["Due by Time"]
            .fillna(freshdesk_raw_df["Created time"])
            # .dt.normalize()
            .sub(freshdesk_raw_df["Resolved time"])
                # .dt.normalize())
            # .dt.days
            .abs()
        )
    ),

    "ENVIRONMENT": (
        freshdesk_raw_df["Company Name"]
        .map(client_environment_df["ENVIRONMENT"])
    ),

    "CLIENT STATUS": (
        freshdesk_raw_df["Company Name"]
        .map(client_environment_df["CLIENT STATE"])
    ),

    "IMPLEMENTATION PHASE": freshdesk_raw_df["Implementation Phase"],

    "SUPPORT LEVEL": (
        freshdesk_raw_df["Support Level"]
        .astype("string")
        .str.extract(r"([123])", expand=False)
        .radd("L")
    )

})

## Exporting

In [ ]:
with pd.ExcelWriter(r"Redmine Freshdesk Dataset Masked.xlsx") as writer:
    redmine_raw_df.to_excel(writer, sheet_name="Redmine Raw Data", index=False)
    redmine_processed_df.to_excel(writer, sheet_name="Redmine Processed Data", index=False)
    freshdesk_raw_df.to_excel(writer, sheet_name="FD Raw", index=False)
    freshdesk_processed_df.to_excel(writer, sheet_name="Freshdesk Processed Data", index=False)
    client_environment_df.to_excel(writer, sheet_name="Client Details")    